# Практическая работа №3 — вариант 7
## HyperNEAT: сеть прямого распространения с одним скрытым слоем

**Среда:** `CartPole-v1` (Gymnasium). **Алгоритм:** HyperNEAT: NEAT эволюционирует CPPN, которая порождает веса геометрически расположенной сети-субстрата. Сеть-субстрат имеет архитектуру **4 → 8 → 2**, ровно один скрытый слой, без рекуррентных и пропускных связей. CPPN имеет собственную изменяемую топологию; это не второй скрытый слой агента.

**Цель:** максимизировать среднюю длину эпизода (до 500 шагов). Успех: среднее ≥475 за 20 независимых оценочных эпизодов. Все бюджеты, seeds и модификации фиксируются. В ноутбуке приведены baseline, абляция, две проверки гиперпараметров, анализ сложности, устойчивости и неудачных режимов.

**Важно:** в ноутбуке используется прямая реализация CPPN/NEAT через `neat-python`. `NEAT` обучает только генератор весов, а не непосредственно веса агента. Это принципиальный признак HyperNEAT.


In [ ]:
%pip -q install neat-python gymnasium[classic-control] numpy pandas scipy matplotlib networkx

In [ ]:
import os, json, random, copy, pickle, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gymnasium as gym
import neat
from scipy import stats
from IPython.display import display
ROOT = Path('hyperneat_variant7_results'); ROOT.mkdir(exist_ok=True)
SEEDS = (7, 17)
print('numpy', np.__version__, 'gymnasium', gym.__version__, 'neat', getattr(neat,'__version__','installed'))

## 1. Формальная постановка

Наблюдение $s=(x,\dot x,\theta,\dot\theta)\in\mathbb R^4$; действия $a\in\{0,1\}$ означают толчок влево или вправо. Награда +1 за каждый прожитый шаг. Fitness — средняя суммарная награда по фиксированному набору начальных состояний (несколько эпизодов). Для честного сравнения все алгоритмы получают одинаковое количество вызовов fitness, а каждый вызов содержит одинаковое количество эпизодов. Дополнительные тестовые оценки считаются отдельно.

**Нормализация:** входные признаки делятся на масштабы `[2.4, 3.0, 0.2095, 3.0]` и ограничиваются диапазоном [-1,1]. Субстрат использует `tanh` в скрытом слое и два линейных выхода; действие выбирается по `argmax`.

**CPPN:** координаты двух нейронов `(x1,y1,x2,y2)` и расстояние `d` подаются в CPPN, результат преобразуется в вес связи. Положение нейронов задаёт регулярность весов. Все связи субстрата — только вход→скрытый и скрытый→выход.


In [ ]:
INPUT_COORDS = np.column_stack((np.linspace(-1,1,4), np.full(4,-1.0)))
HIDDEN_COORDS = np.column_stack((np.linspace(-1,1,8), np.zeros(8)))
OUTPUT_COORDS = np.array([[-0.6,1.0],[0.6,1.0]])

def connection_features(a,b):
    x1,y1 = a; x2,y2 = b
    return [float(x1),float(y1),float(x2),float(y2),float(np.hypot(x2-x1,y2-y1))]

def decode_cppn(cppn, threshold=0.12, weight_scale=3.0):
    def matrix(src,dst):
        m=np.zeros((len(dst),len(src)),dtype=np.float64)
        for i,p in enumerate(dst):
            for j,q in enumerate(src):
                v=float(cppn.activate(connection_features(q,p))[0])
                if abs(v)>=threshold: m[i,j]=np.clip(v,-1,1)*weight_scale
        return m
    return matrix(INPUT_COORDS,HIDDEN_COORDS),matrix(HIDDEN_COORDS,OUTPUT_COORDS)

def act(weights,obs):
    w1,w2=weights
    z=np.clip(np.asarray(obs,dtype=float)/[2.4,3.,.2095,3.],-1,1)
    return int(np.argmax(w2@np.tanh(w1@z)))

def evaluate(weights,seeds,obs_noise=0.0,record=False):
    scores=[]; trajectories=[]
    env=gym.make('CartPole-v1')
    try:
        for seed in seeds:
            ob,_=env.reset(seed=int(seed));rng=np.random.default_rng(int(seed)+991)
            states=[];total=0
            for t in range(500):
                if record:states.append(np.array(ob).copy())
                a=act(weights,ob+rng.normal(0,obs_noise,size=4))
                ob,r,terminated,truncated,_=env.step(a)
                total+=r
                if terminated or truncated:break
            scores.append(float(total))
            if record:trajectories.append(np.array(states))
    finally:env.close()
    return (np.array(scores),trajectories) if record else float(np.mean(scores))

def complexity(weights):
    w1,w2=weights
    return {'connections':int(np.count_nonzero(w1)+np.count_nonzero(w2)),
            'neurons':14,'parameters':int(np.count_nonzero(w1)+np.count_nonzero(w2))}


## 2. NEAT-конфигурация CPPN и бюджет

Входов CPPN — 5, выход — 1. NEAT выполняет мутации весов и структуры (добавление/удаление узлов и связей), скрещивание и видообразование (speciation). В базовом варианте эти механизмы включены. При абляции отключаются **структурные мутации**; все остальные параметры сохраняются.

Для каждого запуска: **популяция 24, 12 поколений, 3 тренировочных эпизода на особь, 288 оценок fitness = 864 эпизода среды**. Для сравнения фиксируем эти величины во всех сериях. Бюджет специально небольшой; при неуспехе он не увеличивается автоматически. `RUN_ALL=True` запускает весь протокол; для быстрой отладки можно поставить `False`.


In [ ]:
CONFIG_TEXT = """[NEAT]
fitness_criterion     = max
fitness_threshold     = 501
pop_size              = 24
reset_on_extinction   = False
no_fitness_termination = True

[DefaultGenome]
num_inputs              = 5
num_outputs             = 1
num_hidden              = 0
feed_forward            = True
initial_connection      = full_direct
activation_default      = tanh
activation_mutate_rate  = 0.0
activation_options      = tanh
aggregation_default     = sum
aggregation_mutate_rate = 0.0
aggregation_options     = sum
bias_init_mean          = 0.0
bias_init_stdev         = 1.0
bias_max_value          = 5.0
bias_min_value          = -5.0
bias_mutate_power       = 0.5
bias_mutate_rate        = 0.7
bias_replace_rate       = 0.1
compatibility_disjoint_coefficient = 1.0
compatibility_weight_coefficient   = 0.5
conn_add_prob           = 0.2
conn_delete_prob        = 0.1
enabled_default         = True
enabled_mutate_rate     = 0.01
node_add_prob           = 0.1
node_delete_prob        = 0.05
node_gene_type          = DefaultNodeGene
conn_gene_type          = DefaultConnectionGene
weight_init_mean        = 0.0
weight_init_stdev       = 1.0
weight_max_value        = 5.0
weight_min_value        = -5.0
weight_mutate_power     = 0.5
weight_mutate_rate      = 0.8
weight_replace_rate     = 0.1
response_init_mean      = 1.0
response_init_stdev     = 0.0
response_max_value      = 5.0
response_min_value      = -5.0
response_mutate_power   = 0.0
response_mutate_rate    = 0.0
response_replace_rate   = 0.0
single_structural_mutation = False
structural_mutation_surer = default

[DefaultSpeciesSet]
compatibility_threshold = 3.0

[DefaultStagnation]
species_fitness_func = max
max_stagnation       = 12
species_elitism      = 1

[DefaultReproduction]
elitism            = 1
survival_threshold = 0.25
min_species_size   = 2
"""
CONFIG_PATH=ROOT/'cppn_neat.ini';CONFIG_PATH.write_text(CONFIG_TEXT,encoding='utf-8')

def load_config():
    return neat.Config(neat.DefaultGenome,neat.DefaultReproduction,
                       neat.DefaultSpeciesSet,neat.DefaultStagnation,str(CONFIG_PATH))

@dataclass
class RunSettings:
    pop_size:int=24
    generations:int=12
    episodes_per_fitness:int=3
    threshold:float=0.12
    weight_scale:float=3.0
    seed:int=7
    structural_mutations:bool=True

BASE=RunSettings()
RUN_ALL=True
print(asdict(BASE))

In [ ]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed)

def train_hyperneat(settings, label='HyperNEAT'):
    seed_everything(settings.seed)
    config=load_config()
    config.pop_size=settings.pop_size
    if not settings.structural_mutations:
        config.genome_config.node_add_prob=0.
        config.genome_config.node_delete_prob=0.
        config.genome_config.conn_add_prob=0.
        config.genome_config.conn_delete_prob=0.
    population=neat.Population(config)
    history=[];best_seen=None;best_score=-np.inf;best_gen=0
    # Единый набор тренировочных seeds на поколение для всех особей.
    # Не используем тестовые seeds при выборе победителя.
    def fitness_fn(genomes,config):
        nonlocal best_seen,best_score,best_gen
        generation=len(history)
        epseeds=[settings.seed*100000+generation*100+e for e in range(settings.episodes_per_fitness)]
        values=[];weights_list=[]
        for gid,genome in genomes:
            cppn=neat.nn.FeedForwardNetwork.create(genome,config)
            weights=decode_cppn(cppn,settings.threshold,settings.weight_scale)
            score=evaluate(weights,epseeds)
            genome.fitness=score;values.append(score);weights_list.append(weights)
            if score>best_score:
                best_score=score;best_seen=copy.deepcopy(genome);best_gen=generation
        ix=int(np.argmax(values)); g=genomes[ix][1];w=weights_list[ix]
        history.append({'generation':generation,'best':float(np.max(values)),
                        'mean':float(np.mean(values)), 'cppn_nodes':len(g.nodes),
                        'cppn_connections':sum(c.enabled for c in g.connections.values()),
                        **complexity(w)})
    winner=population.run(fitness_fn,settings.generations)
    # Выбираем лучшую особь среди всех оценённых поколений, а не только последнего.
    genome=best_seen if best_seen is not None else winner
    cppn=neat.nn.FeedForwardNetwork.create(genome,config)
    weights=decode_cppn(cppn,settings.threshold,settings.weight_scale)
    return {'name':label,'settings':asdict(settings),'history':pd.DataFrame(history),
            'weights':weights,'genome':genome,'best_train':best_score,'best_generation':best_gen}

def train_random(settings,label='Random search'):
    seed_everything(settings.seed)
    rng=np.random.default_rng(settings.seed)
    history=[];best=-np.inf;best_weights=None
    for g in range(settings.generations):
        epseeds=[settings.seed*100000+g*100+e for e in range(settings.episodes_per_fitness)]
        values=[]
        for _ in range(settings.pop_size):
            weights=(rng.normal(0,settings.weight_scale/2,(8,4)),
                     rng.normal(0,settings.weight_scale/2,(2,8)))
            score=evaluate(weights,epseeds)
            values.append(score)
            if score>best:best=score;best_weights=tuple(w.copy() for w in weights)
        history.append({'generation':g,'best':max(values),'mean':np.mean(values),
                        'cppn_nodes':np.nan,'cppn_connections':np.nan,**complexity(best_weights)})
    return {'name':label,'settings':asdict(settings),'history':pd.DataFrame(history),
            'weights':best_weights,'genome':None,'best_train':best}


## 3. Контролируемые эксперименты

Серии: два независимых запуска основного HyperNEAT; baseline `random search` с тем же бюджетом; абляция без структурных мутаций; два сравнения по одному параметру (`threshold=0.0` вместо `0.12` и `weight_scale=1.5` вместо `3.0`). Каждый эксперимент сохраняет все остальные параметры и seed. Тестовые эпизоды отделены от тренировочных.


In [ ]:
EXPERIMENTS=[
    ('HyperNEAT seed 7','hyper',replace(BASE,seed=7)),
    ('HyperNEAT seed 17','hyper',replace(BASE,seed=17)),
    ('Random search','random',replace(BASE,seed=7)),
    ('Ablation: no structural mutation','hyper',replace(BASE,seed=7,structural_mutations=False)),
    ('Hyperparameter: threshold 0','hyper',replace(BASE,seed=7,threshold=0.0)),
    ('Hyperparameter: weight scale 1.5','hyper',replace(BASE,seed=7,weight_scale=1.5)),
]
results={}
for label,kind,settings in EXPERIMENTS if RUN_ALL else EXPERIMENTS[:1]:
    print('Training:',label,flush=True)
    start=time.time()
    result=train_hyperneat(settings,label) if kind=='hyper' else train_random(settings,label)
    results[label]=result
    print('best training fitness:',round(result['best_train'],2),'elapsed:',round(time.time()-start,1),'s')
    with open(ROOT/(label.replace(' ','_').replace(':','')+'.pkl'),'wb') as f:
        pickle.dump(result,f)
(ROOT/'experiment_config.json').write_text(json.dumps([{'label':x,'method':y,**asdict(z)} for x,y,z in EXPERIMENTS],indent=2),encoding='utf-8')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(14,5))
for label,r in results.items():
    h=r['history'];axs[0].plot(h.generation,h.best,label=label)
    axs[1].plot(h.generation,h['mean'],label=label)
for ax,title in zip(axs,['Best fitness by generation','Mean fitness by generation']):
    ax.set(title=title,xlabel='Generation',ylabel='Episode return');ax.grid(alpha=.3)
axs[1].legend(fontsize=8,bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout();plt.savefig(ROOT/'learning_curves.png',dpi=160);plt.show()

## 4. Оценка на новых начальных состояниях и устойчивость

Проверяем 30 тестовых эпизодов для каждой модели при `obs_noise=0` и при `obs_noise=0.08`. Изменён **ровно один фактор**: добавлен гауссов шум к наблюдениям, остальные параметры среды и политика неизменны. Используются одинаковые тестовые seeds. Статистическое сравнение — парный t-test между HyperNEAT и baseline; тест служит описательной проверкой, поскольку эпизоды внутри одного обучения не заменяют независимые обучающие запуски.


In [ ]:
TEST_SEEDS=list(range(90000,90030)); rows=[]; episode_scores={}
for label,r in results.items():
    for noise,scenario in [(0.,'base'),(.08,'noisy observations')]:
        scores,_=evaluate(r['weights'],TEST_SEEDS,obs_noise=noise,record=True)
        episode_scores[(label,scenario)]=scores
        rows.append({'method':label,'scenario':scenario,'mean_reward':scores.mean(),
                     'std_reward':scores.std(ddof=1),'success_rate_%':100*np.mean(scores>=475),
                     'fitness_evaluations':r['settings']['pop_size']*r['settings']['generations'],
                     'training_episodes':r['settings']['pop_size']*r['settings']['generations']*r['settings']['episodes_per_fitness'],
                     'seed':r['settings']['seed']})
summary=pd.DataFrame(rows);display(summary.round(2));summary.to_csv(ROOT/'evaluation_summary.csv',index=False)
if ('HyperNEAT seed 7','base') in episode_scores and ('Random search','base') in episode_scores:
    a=episode_scores[('HyperNEAT seed 7','base')];b=episode_scores[('Random search','base')]
    print('Paired t-test, HyperNEAT vs Random search:',stats.ttest_rel(a,b))
    print('Note: p-value is exploratory; independent training seeds are needed for generalization claims.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(14,4))
for label,r in results.items():
    h=r['history'];axs[0].plot(h.generation,h.cppn_nodes,label=label)
    axs[1].plot(h.generation,h.cppn_connections,label=label)
axs[0].set(title='CPPN nodes',xlabel='Generation',ylabel='Count')
axs[1].set(title='CPPN enabled connections',xlabel='Generation',ylabel='Count')
for ax in axs:ax.grid(alpha=.3)
axs[1].legend(fontsize=7,bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout();plt.savefig(ROOT/'cppn_complexity.png',dpi=160);plt.show()
# Сложность порождённого субстрата — отдельная характеристика
fig,ax=plt.subplots(figsize=(9,4))
for label,r in results.items():ax.plot(r['history'].generation,r['history'].connections,label=label)
ax.set(title='Active substrate connections (max 48)',xlabel='Generation',ylabel='Connections');ax.grid(alpha=.3)
ax.legend(fontsize=7,bbox_to_anchor=(1.02,1),loc='upper left');plt.tight_layout();plt.show()

## 5. Визуализация структуры и поведения

Для лучшей модели рисуем геометрический субстрат и траектории состояния в базовой и модифицированной постановках. У HyperNEAT веса зависят от геометрии; граф отображает связи с ненулевыми весами. Для анализа **ранней, промежуточной и поздней** структуры отдельно строятся кривые числа узлов и связей CPPN на каждом поколении; это не подмена CPPN субстратом.


In [ ]:
def plot_substrate(weights,title):
    w1,w2=weights
    fig,ax=plt.subplots(figsize=(9,5))
    layers=[INPUT_COORDS,HIDDEN_COORDS,OUTPUT_COORDS]
    for layer in layers:ax.scatter(layer[:,0]*2+len(layer)*0,layer[:,1],s=70,zorder=3)
    # Используем горизонтальную координату как слой, вертикальную как положение в слое
    positions=[[(0,y) for y in np.linspace(-1,1,4)],[(1,y) for y in np.linspace(-1,1,8)],[(2,y) for y in np.linspace(-1,1,2)]]
    for lay,mat in enumerate([w1,w2]):
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if mat[i,j]!=0:
                    x1,y1=positions[lay][j];x2,y2=positions[lay+1][i]
                    ax.plot([x1,x2],[y1,y2],color='tab:blue' if mat[i,j]>0 else 'tab:red',alpha=min(1,.2+abs(mat[i,j])/5),lw=.5+abs(mat[i,j])/2)
    for p in positions:
        for x,y in p:ax.scatter(x,y,s=110,c='white',edgecolors='black',zorder=4)
    ax.set(xlim=(-.2,2.2),ylim=(-1.3,1.3),xticks=[0,1,2],xticklabels=['Inputs (4)','Hidden (8)','Outputs (2)'],title=title);ax.axis('on');plt.show()

if 'HyperNEAT seed 7' in results:
    chosen=results['HyperNEAT seed 7'];plot_substrate(chosen['weights'],'HyperNEAT substrate: blue positive, red negative')
    fig,axs=plt.subplots(1,2,figsize=(13,4))
    for ax,noise,title in zip(axs,[0,.08],['Base environment','Noisy observations']):
        scores,trajs=evaluate(chosen['weights'],[90101],obs_noise=noise,record=True)
        ax.plot(trajs[0][:,0],label='cart position');ax.plot(trajs[0][:,2],label='pole angle')
        ax.set(title=f'{title}: reward={scores[0]:.0f}',xlabel='Time step',ylabel='Observation');ax.legend()
    plt.tight_layout();plt.savefig(ROOT/'trajectories.png',dpi=160);plt.show()

## 6. Неудачные режимы и выводы

Нужно анализировать не менее двух случаев деградации. Ниже автоматически сравниваются: (1) отключение структурных мутаций и (2) шум наблюдений. Если ухудшения нет, это фиксируется как отрицательный результат, а не объявляется деградацией. Дополнительный режим — увеличение порога экспрессии, что может обнулить большинство связей (демонстрация с неизменёнными весами CPPN).


In [ ]:
if 'HyperNEAT seed 7' in results:
    ref=summary[(summary.method=='HyperNEAT seed 7')&(summary.scenario=='base')].iloc[0].mean_reward
    for label in ['Ablation: no structural mutation','Hyperparameter: threshold 0','Hyperparameter: weight scale 1.5']:
        sub=summary[(summary.method==label)&(summary.scenario=='base')]
        if len(sub):
            diff=sub.iloc[0].mean_reward-ref
            print(f'{label}: delta reward {diff:+.2f}; '+('degradation' if diff<0 else 'no degradation observed'))
    noisy=summary[(summary.method=='HyperNEAT seed 7')&(summary.scenario=='noisy observations')].iloc[0].mean_reward
    print(f'Observation noise: delta reward {noisy-ref:+.2f}')
    # Неудачный режим: отключение значительной части связей через высокий порог
    cfg=load_config();cp=neat.nn.FeedForwardNetwork.create(results['HyperNEAT seed 7']['genome'],cfg)
    sparse=decode_cppn(cp,threshold=.95,weight_scale=3.)
    print('High expression threshold .95: active connections',complexity(sparse)['connections'],
          'mean reward',evaluate(sparse,TEST_SEEDS))
print('Conclusions must be written using actual values in evaluation_summary.csv, not invented results.')

## 7. Сохранение, загрузка, проверка воспроизводимости

Сохраняем **геном CPPN, конфигурацию, порождённые веса и метаданные**. При загрузке проверяем совпадение поведения на фиксированных seeds. Отдельно сохраняется README с командами запуска и JSON параметров. Экспорт файлов возможен через панель Files в Colab.


In [ ]:
if 'HyperNEAT seed 7' in results:
    r=results['HyperNEAT seed 7']
    path=ROOT/'hyperneat_best.pkl'
    with path.open('wb') as f:pickle.dump({'genome':r['genome'],'settings':r['settings'],'weights':r['weights']},f)
    with path.open('rb') as f:restored=pickle.load(f)
    a=evaluate(r['weights'],[12001,12002,12003]);b=evaluate(restored['weights'],[12001,12002,12003])
    assert a==b, (a,b)
    print('Reload verification OK:',a)
README="""# HyperNEAT — вариант 7

1. Откройте `Lab_3_HyperNEAT_variant7.ipynb` в Google Colab.
2. Выполните `Runtime → Run all` (в ноутбуке RUN_ALL=True).
3. Для повторения экспериментов используйте SEEDS=(7,17) и параметры RunSettings.
4. Результаты: папка `hyperneat_variant7_results` (CSV, PNG, PKL, JSON, INI).
5. Сохранённая модель `hyperneat_best.pkl` загружается Python/pickle (только из доверенного источника).
6. Для проверки повторяемости сравните evaluation_summary.csv и learning_curves.png.

Обучение выполняется без GPU, время зависит от CPU Colab.
"""
(ROOT/'README.md').write_text(README,encoding='utf-8')
print('Files:',[p.name for p in ROOT.iterdir()])

## 8. Интерпретация и ограничения

- Сравнение основано на **одинаковом количестве вызовов fitness**, а не на одинаковом числе элементарных операций: генерация CPPN и NEAT дороже random search.
- Кривые fitness на обучающих seeds оптимистичны; основной результат — независимый тест.
- Два seed обеспечивают минимальную проверку воспроизводимости, но недостаточны для сильных статистических выводов о преимуществах метода.
- В отчёте приложить таблицу `evaluation_summary.csv`, графики обучения/сложности, визуализацию субстрата и поведения, а также объяснить случаи деградации, подтверждённые измерениями.
- `pickle` загружать только из доверенных файлов. На другой машине версии зависимостей могут влиять на воспроизводимость.
